# How LLMs Actually Work

### A from-scratch walkthrough of what happens when you ask a question

This notebook builds up the ideas behind Large Language Models step by step, with runnable code. No GPU needed — everything here runs on CPU in seconds, because the goal is to *understand the mechanism*, not to train a competitive model.

**What we'll cover:**

1. The one-sentence answer: what an LLM actually does
2. Tokenization — turning text into numbers
3. Embeddings — turning numbers into meaning
4. Attention — the core idea that made transformers work
5. The transformer block — putting it together
6. Next-token prediction — how "thinking" is actually sampling
7. The full inference loop — what happens when you hit *enter*
8. Training vs. inference, and how models differ from each other

You don't need deep math to follow this. Where math appears, it's explained in words first.


## 0. Setup

We only use `numpy` for the core mechanics (so nothing is hidden inside a framework) and `torch` for the parts where reimplementing would be tedious. If `torch` isn't installed, the numpy sections still run standalone.


In [1]:
import numpy as np
np.random.seed(0)

try:
    import torch
    import torch.nn.functional as F
    HAS_TORCH = True
    print("torch available:", torch.__version__)
except ImportError:
    HAS_TORCH = False
    print("torch not available — numpy sections still run")


torch not available — numpy sections still run


## 1. The one-sentence answer

> **An LLM is a function that reads a sequence of words and predicts the next word. That's it.**

Everything else — answering questions, writing code, holding a conversation — is that single ability applied over and over. You give it *"The capital of France is"* and it assigns a high probability to *"Paris"*. To answer a question, it just keeps predicting the next word, feeding each new word back in, until it decides to stop.

The entire field is about making that next-word prediction *very, very good*. The rest of this notebook unpacks how.


In [2]:
# A cartoon of the whole thing before we build the real version.
# Given some context, produce a probability over possible next words.

vocab = ["Paris", "London", "cheese", "the", "is"]

def toy_llm(context):
    # A real model computes these numbers. Here we hard-code them to show the shape.
    if context.strip().endswith("capital of France is"):
        logits = np.array([5.0, 1.0, 0.2, 0.1, 0.1])   # "Paris" scores highest
    else:
        logits = np.array([1.0, 1.0, 1.0, 1.0, 1.0])
    probs = np.exp(logits) / np.exp(logits).sum()       # softmax -> probabilities
    return dict(zip(vocab, probs.round(3)))

toy_llm("The capital of France is")


{'Paris': np.float64(0.96),
 'London': np.float64(0.018),
 'cheese': np.float64(0.008),
 'the': np.float64(0.007),
 'is': np.float64(0.007)}

Notice the last step: `softmax`. It turns arbitrary scores (*logits*) into probabilities that sum to 1. This exact operation appears at the output of every LLM. Hold onto it.

## 2. Tokenization — text into numbers

Models don't see letters or words. They see **token IDs** — integers. A *tokenizer* chops text into chunks (tokens) and maps each to an ID. Modern LLMs use *subword* tokenization: common words become one token, rare words split into pieces. This is why the model can handle words it has never seen — it assembles them from parts.

Below is a tiny word-level tokenizer to show the idea. Real tokenizers (like BPE, used by GPT models) work on subwords, but the principle is identical: **text ↔ list of integers.**


In [3]:
corpus = "the cat sat on the mat the dog sat on the log"
words = corpus.split()

# Build a vocabulary: unique tokens -> integer IDs
vocab = sorted(set(words))
stoi = {w: i for i, w in enumerate(vocab)}   # string -> id
itos = {i: w for w, i in stoi.items()}       # id -> string

def encode(text): return [stoi[w] for w in text.split()]
def decode(ids):  return " ".join(itos[i] for i in ids)

print("Vocabulary:", stoi)
print("Encoded:   ", encode("the cat sat"))
print("Decoded:   ", decode([6, 0, 5]))


Vocabulary: {'cat': 0, 'dog': 1, 'log': 2, 'mat': 3, 'on': 4, 'sat': 5, 'the': 6}
Encoded:    [6, 0, 5]
Decoded:    the cat sat


**Why subwords in real models?** The word `tokenization` might split into `token` + `ization`. That means the model reuses what it learned about `token` and about the `-ization` suffix, instead of treating every long word as a brand-new symbol. It keeps the vocabulary small (~50k–100k tokens) while covering essentially any text, including typos and code.

## 3. Embeddings — numbers into meaning

A token ID like `42` carries no meaning — it's just an index. The first thing the model does is look up a **vector** for each token: a list of, say, 768 numbers. This is the *embedding*. These vectors are *learned* during training so that tokens used in similar ways end up close together in this high-dimensional space.

This is the first place "meaning" enters the model: **direction and distance in embedding space encode relationships.**


In [4]:
embedding_dim = 8   # tiny, for illustration; real models use 768 - 12288+
vocab_size = len(vocab)

# The embedding table: one learnable vector per token. Random here; learned in practice.
embedding_table = np.random.randn(vocab_size, embedding_dim) * 0.1

def embed(ids):
    return embedding_table[ids]   # shape: (num_tokens, embedding_dim)

x = embed(encode("the cat sat"))
print("Three tokens, each an 8-dim vector:")
print(x.round(2))


Three tokens, each an 8-dim vector:
[[-0.16 -0.02 -0.09  0.04 -0.05 -0.12 -0.    0.04]
 [ 0.18  0.04  0.1   0.22  0.19 -0.1   0.1  -0.02]
 [-0.1  -0.14 -0.17  0.2  -0.05 -0.04 -0.13  0.08]]


Because vectors have direction, we can measure how "similar" two tokens are with **cosine similarity**. After real training, `cat` and `dog` would be closer to each other than either is to `the`. With our random table the numbers are meaningless — but the *mechanism* is exactly what real models use.

In [5]:
def cosine(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

cat = embedding_table[stoi["cat"]]
dog = embedding_table[stoi["dog"]]
the = embedding_table[stoi["the"]]
print(f"cos(cat, dog) = {cosine(cat, dog): .3f}")
print(f"cos(cat, the) = {cosine(cat, the): .3f}")
print("(random values here — after training, cat~dog would be higher)")


cos(cat, dog) =  0.737
cos(cat, the) = -0.318
(random values here — after training, cat~dog would be higher)


## 4. Attention — the idea that made it all work

Here is the heart of the transformer, and the single most important concept in the notebook.

**The problem:** the meaning of a word depends on the words around it. In *"the bank of the river"* vs *"money in the bank"*, `bank` means different things. The model needs each word to **look at** the other words and pull in relevant context.

**The solution — self-attention:** every token produces three vectors:

- a **Query** (Q): *"what am I looking for?"*
- a **Key** (K): *"what do I contain?"*
- a **Value** (V): *"what do I pass along if attended to?"*

Each token compares its Query against every token's Key (a dot product = "how relevant is this other word to me?"), turns those scores into weights with softmax, and takes a weighted sum of the Values. The result: each token's representation is now a *blend* of the tokens most relevant to it.

This is the mechanism behind *"attention is all you need."* Let's build it.


In [6]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def self_attention(X, W_q, W_k, W_v, causal=True):
    Q = X @ W_q          # (T, d) what each token looks for
    K = X @ W_k          # (T, d) what each token offers
    V = X @ W_v          # (T, d) what each token passes on
    d = Q.shape[-1]

    scores = Q @ K.T / np.sqrt(d)             # (T, T) relevance of every token to every token

    if causal:
        # A token may only attend to itself and earlier tokens — it can't see the future.
        # This is what makes generation left-to-right.
        T = scores.shape[0]
        mask = np.triu(np.ones((T, T)), k=1).astype(bool)
        scores[mask] = -np.inf

    weights = softmax(scores, axis=-1)        # (T, T) attention weights, each row sums to 1
    out = weights @ V                         # (T, d) context-mixed representation
    return out, weights


In [7]:
# Run attention on our 3-token example
T, d = x.shape
W_q = np.random.randn(d, d) * 0.3
W_k = np.random.randn(d, d) * 0.3
W_v = np.random.randn(d, d) * 0.3

out, weights = self_attention(x, W_q, W_k, W_v, causal=True)

print("Attention weights (row = token attending, col = token attended to):")
tokens = "the cat sat".split()
print("           " + "  ".join(f"{t:>5}" for t in tokens))
for i, t in enumerate(tokens):
    print(f"{t:>5} ->  " + "  ".join(f"{w:5.2f}" for w in weights[i]))


Attention weights (row = token attending, col = token attended to):
             the    cat    sat
  the ->   1.00   0.00   0.00
  cat ->   0.51   0.49   0.00
  sat ->   0.34   0.33   0.33


Look at the weight matrix. It's **lower-triangular** — the zeros in the upper right are the *causal mask* in action. The token `the` (first) can only attend to itself. `sat` (third) can attend to all three. This is precisely why an LLM generates text left to right: when predicting the next word, it is not allowed to peek at words that haven't been generated yet.

**Multi-head attention:** real models run several of these attention computations in parallel ("heads"), each free to focus on a different kind of relationship — one head might track subject–verb agreement, another might track long-range references. Their outputs are concatenated. The code is the same, just stacked.


## 5. The transformer block

A transformer is just this pattern, repeated N times:

```
for each block:
    x = x + attention(layernorm(x))     # mix information across tokens
    x = x + feedforward(layernorm(x))   # process each token individually
```

Two sub-parts:
- **Attention** moves information *between* tokens (Section 4).
- **Feed-forward network (FFN)** is a small MLP applied to each token *independently* — this is where a lot of factual "knowledge" is stored.

Two supporting tricks:
- **Residual connections** (`x = x + ...`): let the original signal flow through, which makes very deep stacks trainable.
- **Layer normalization**: keeps the numbers in a healthy range so training stays stable.

Stack 12, 32, or 100+ of these blocks and you have GPT-style architecture. Below is one full block in numpy.


In [8]:
def layernorm(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def feedforward(x, W1, W2):
    h = np.maximum(0, x @ W1)   # ReLU expands to a larger hidden size...
    return h @ W2               # ...then projects back down

def transformer_block(x, params):
    # Sub-layer 1: attention with a residual connection
    a, _ = self_attention(layernorm(x), params['Wq'], params['Wk'], params['Wv'])
    x = x + a
    # Sub-layer 2: feed-forward with a residual connection
    f = feedforward(layernorm(x), params['W1'], params['W2'])
    x = x + f
    return x

hidden = 4 * d
params = dict(
    Wq=np.random.randn(d, d)*0.3, Wk=np.random.randn(d, d)*0.3, Wv=np.random.randn(d, d)*0.3,
    W1=np.random.randn(d, hidden)*0.3, W2=np.random.randn(hidden, d)*0.3,
)

y = transformer_block(x, params)
print("Input shape: ", x.shape, " Output shape:", y.shape, " (same shape -> blocks stack cleanly)")


Input shape:  (3, 8)  Output shape: (3, 8)  (same shape -> blocks stack cleanly)


The output has the *same shape* as the input. That's deliberate: it means you can feed the output of one block straight into the next, stacking as deep as you like. Depth is where capability comes from.

## 6. From vectors to a next word

After the final block, each token position holds a context-rich vector. To predict the next token, the model projects that vector back to *vocabulary size* — one score (logit) per possible token — and applies **softmax** to get probabilities. Same softmax from Section 1, now earned.


In [9]:
# Project the last token's vector to a score for every word in the vocab
W_out = np.random.randn(d, vocab_size) * 0.3
last_vector = y[-1]                     # representation at the final position
logits = last_vector @ W_out
probs = softmax(logits)

print("Predicted next-word probabilities:")
for tok, p in sorted(zip(vocab, probs), key=lambda z: -z[1]):
    print(f"  {tok:>5}: {p:.3f}")


Predicted next-word probabilities:
    dog: 0.207
    the: 0.159
    cat: 0.151
    log: 0.148
    sat: 0.139
     on: 0.117
    mat: 0.078


**How the next word is chosen** — this is what people mean by an LLM's "thinking," and it's really just *sampling* from this distribution:

- **Greedy**: always take the highest-probability token. Deterministic, can be repetitive.
- **Temperature**: divide logits by a number `T` before softmax. `T<1` sharpens (more confident, safer); `T>1` flattens (more random, more creative).
- **Top-k / top-p**: only sample from the most likely handful of tokens, discarding the long tail of nonsense.

This is the knob behind "make the model more creative" vs "more deterministic."


In [10]:
def sample_next(logits, temperature=1.0, top_k=None):
    logits = logits / temperature
    if top_k is not None:
        cutoff = np.sort(logits)[-top_k]
        logits = np.where(logits < cutoff, -np.inf, logits)
    p = softmax(logits)
    return np.random.choice(len(p), p=p)

print("temp=0.5 (focused): ", [vocab[sample_next(logits, 0.5)] for _ in range(8)])
print("temp=1.5 (creative):", [vocab[sample_next(logits, 1.5)] for _ in range(8)])


temp=0.5 (focused):  ['the', 'dog', 'on', 'dog', 'sat', 'log', 'dog', 'dog']
temp=1.5 (creative): ['dog', 'mat', 'on', 'the', 'on', 'sat', 'dog', 'log']


## 7. What actually happens when you ask a question

Now we can trace the whole path, end to end. When you type a prompt and hit enter:

1. **Tokenize** — your text becomes a list of token IDs.
2. **Embed** — each ID becomes a vector; position information is added so the model knows word order.
3. **Forward pass** — the sequence flows through every transformer block. Attention mixes context; feed-forward layers apply learned knowledge.
4. **Predict** — the final vector is projected to logits over the whole vocabulary; softmax gives probabilities.
5. **Sample** — one token is chosen (greedy / temperature / top-p).
6. **Append and repeat** — the chosen token is added to the sequence, and the model runs again to predict the *next* one. This is called **autoregressive** generation.
7. **Stop** — generation ends when the model emits a special end-of-text token or hits a length limit.

The loop below is the real thing in miniature — this exact structure runs inside ChatGPT, Claude, and every other LLM.


In [11]:
def generate(prompt_ids, all_params, n_new=5, temperature=0.8):
    ids = list(prompt_ids)
    for _ in range(n_new):
        # 1-2. embed the current sequence (+ a simple positional signal)
        h = embedding_table[ids]
        h = h + np.arange(len(ids))[:, None] * 0.01   # toy positional encoding
        # 3. forward pass through the block(s)
        h = transformer_block(h, all_params)
        # 4. logits for the LAST position only (that's what predicts the next token)
        logits = h[-1] @ W_out
        # 5. sample
        nxt = sample_next(logits, temperature=temperature)
        # 6. append and loop
        ids.append(nxt)
    return ids

prompt = encode("the cat")
out_ids = generate(prompt, params, n_new=5)
print("Prompt:   ", decode(prompt))
print("Generated:", decode(out_ids))
print("\n(Output is gibberish — the model is untrained. The *process* is exactly right.)")


Prompt:    the cat
Generated: the cat log the mat log log

(Output is gibberish — the model is untrained. The *process* is exactly right.)


The text is nonsense because none of our weights were trained — they're random. But every step is the genuine article. **Training** is the missing ingredient: showing the model billions of sentences and nudging all those weight matrices (`W_q`, `W_k`, `W1`, embeddings, ...) so that its predicted next-token probabilities match reality. That nudging is done with **backpropagation + gradient descent** — the same core algorithm behind all deep learning.

## 8. Training in one picture, and why models differ

**Training objective.** It's almost embarrassingly simple: take real text, hide the next word, ask the model to predict it, and penalize it when it's wrong. The penalty is *cross-entropy loss* — high when the model put low probability on the true next word. Repeat across a huge corpus, adjusting weights via gradient descent, and the model gradually learns grammar, facts, reasoning patterns, and style — all as a side effect of getting good at "guess the next token."


In [12]:
def cross_entropy(probs, true_id):
    return -np.log(probs[true_id] + 1e-9)

demo = softmax(np.array([2.0, 1.0, 0.1]))
print("If the true next token is #0 (high prob):", round(cross_entropy(demo, 0), 3), "-> low loss")
print("If the true next token is #2 (low prob): ", round(cross_entropy(demo, 2), 3), "-> high loss")
print("The optimizer pushes weights to make the correct token's probability higher.")


If the true next token is #0 (high prob): 0.417 -> low loss
If the true next token is #2 (low prob):  2.317 -> high loss
The optimizer pushes weights to make the correct token's probability higher.


**The three stages of a modern chat model:**

1. **Pretraining** — predict the next token across trillions of tokens of internet/text. Produces a *base model* with broad knowledge but no instinct to be helpful.
2. **Supervised fine-tuning (SFT)** — train on curated *instruction → good response* examples so it learns to follow instructions and answer questions.
3. **Preference tuning (RLHF / DPO)** — humans rank responses; the model is tuned to prefer the responses people rate higher. This is what makes a model feel helpful, harmless, and coherent in conversation.

**Why do LLMs differ from one another?** Same core recipe, different choices:

| Factor | What it changes |
|---|---|
| **Parameters** (7B vs 70B vs 400B+) | Raw capacity — bigger tends to mean more capable, at higher cost |
| **Training data** | What it knows, its blind spots, its languages and domains |
| **Context length** | How much text it can consider at once (a few pages vs a whole book) |
| **Fine-tuning & preference data** | Personality, helpfulness, safety, refusal behavior |
| **Architecture tweaks** | e.g. Mixture-of-Experts, which activates only part of the network per token for efficiency |
| **Tokenizer** | How efficiently it handles code, math, and non-English text |

Two models can share the identical transformer skeleton you built above and still behave very differently because of these.


## 9. The same idea, in PyTorch

To connect the numpy version to the real world, here's a compact but *complete* GPT-style block in PyTorch. It's the same three ideas — attention, feed-forward, residuals — just with autograd so it's actually trainable.


## Recap

- An LLM is a **next-token predictor**. Everything it does is that ability, looped.
- **Tokenization** turns text into integers; **embeddings** turn integers into meaningful vectors.
- **Attention** lets each token pull in context from other tokens — the key invention.
- A **transformer block** = attention + feed-forward + residuals + normalization, stacked deep.
- Generation is **autoregressive**: predict, sample, append, repeat.
- **Training** = getting good at next-token prediction via backpropagation; **chat behavior** comes from later fine-tuning and preference tuning.
- Models differ through **size, data, context length, tuning, and architecture** — not usually through a different core mechanism.

### Where to go next
- Swap the toy corpus for a real text file and actually *train* the `TinyGPT` — watch the loss drop and the samples become coherent.
- Visualize the attention weight matrices as heatmaps to see which tokens attend to which.
- Implement byte-pair encoding to replace the word-level tokenizer.
- Read Karpathy's *"Let's build GPT"* and the original *"Attention Is All You Need"* paper alongside this notebook.
